# Structural Continuity Analysis of Battery Tomographies — Generation 2: Geometry-Independent Model

## Introduction

The internal structure of cylindrical batteries consists of thin, tightly wound layers (current collectors, electrodes, and separators) forming a spiral (often referred to as a "jelly roll"). When evaluating the performance of Machine Learning denoising models on neutron tomography data, simple linear checks are insufficient because the layers are inherently curved.

This notebook implements the **second-generation** structural-continuity pipeline. Like the first-generation notebook, its goal is to turn each circular battery cross-section into a representation where every physical winding is easy to check for breaks, merges, or other discontinuities, so this can be done automatically and quantitatively across an entire tomography volume.

The key difference from Generation 1 is what "flattening" a winding means here: instead of assuming every winding follows the same constant-pitch spiral and de-spiralling the whole image with one global correction, this version traces **each winding individually**, directly from its own connected shape in a plain (angle vs. radius) unwrap — with no assumption about pitch, spacing, or regularity at all. This makes it substantially more robust to the local irregularities (pitch drift, non-uniform spacing, distortion near the center) that Generation 1's constant-pitch model struggles with.

## The Model

This pipeline is **geometry-independent**: it makes no assumption that the winding follows any particular curve (Archimedean or otherwise). Instead:

1. The image is resampled onto a plain (angle, radius) polar grid (`build_polar_binary`) with **no pitch correction applied** — so, unlike Generation 1, each winding appears as a **diagonal** band, not a horizontal one.
2. The polar image is background-subtracted and adaptively thresholded (same idea as Generation 1: a local Otsu threshold, radius by radius) to produce a binary image.
3. Each winding is identified as its own **connected component** in that binary image (`trace_windings_via_components`) — an 8-connected diagonal band that covers (almost) the full 360°. A region is only accepted as a genuine winding if it spans at least `min_angular_coverage_fraction` (90% by default) of the full circle in one connected piece.
4. Each accepted winding is converted from a sparse `{angle: radius}` mapping into a dense, per-degree array (`densify_track`), using circular interpolation to bridge any small gaps.

Because no functional form is fitted to the winding at all, this model adapts automatically to whatever pitch and spacing are actually present — at the cost of the different failure mode described below.

## Limitations

- **Scope: geometry and winding-tracing only.** As delivered, this notebook detects the battery's geometry (center, inner/outer radius) and traces each individual winding — it does **not** yet implement the fissure/fusion defect classification, full-volume batch loop, CSV export, or quantitative ground-truth validation that Generation 1 has (see Phases 3-7 there). A broken or merged winding here is not reported as a *located* defect; see the next point.
- **Broken windings are dropped, not reported as defects.** A winding is only kept if its connected region covers at least `min_angular_coverage_fraction` (90%) of the full circle. If a real defect breaks a winding's connectivity by more than that, the (now two) fragments each fall under the threshold and are **both silently discarded** — the winding simply disappears from the count rather than being flagged, located, and sized the way Generation 1's `detect_discontinuities` does. This is also the main reason the algorithm is reliable on "clean" intermediate slices but degrades near the ends of the cell: a large, irregular dark region there can break several windings' connectivity at once, so only the untouched ones are still counted — for example, at one slice near the cell's end, only 2 windings were traced despite dozens being physically present, because a large irregular dark region broke connectivity across most of the winding set.
- **Hollow-core detection can misfire on unusual contrast.** `detect_spiral_inner_radius` assumes the angular variation of the residual signal rises from a flat noise floor to a clear plateau once real winding structure begins; a slice with unusually low contrast or an atypical core could produce an unreliable transition point.
- **Outer-boundary detection assumes the strongest edge in the image is the casing.** `detect_max_radius` picks out the radius with the largest local intensity gradient at each angle; a brighter or sharper artefact elsewhere in the field of view could in principle be mistaken for the boundary.
- **Empirically tuned constants.** `min_area`, `min_angular_coverage_fraction`, `outlier_tolerance`, and the noise-floor/plateau sampling windows in `detect_spiral_inner_radius` (radii 0-40 px and 150-400 px) were calibrated on this dataset's geometry and resolution, and will likely need re-tuning for a different cell or imaging setup.
- **No quantitative validation yet.** Unlike Generation 1 (which checks its detections against a small, manually confirmed list), this notebook currently has no ground-truth comparison — its only verification is the visual sanity check in Phase 5.
- **Demonstration harness, not a batch pipeline.** The current execution cell processes four individually named slice files rather than looping over a full stack; extending it to a full volume (as Generation 1 does) is straightforward but not yet implemented here.

## Attribution

This codebase was developed by Genar Iturri Bello during a summer internship at the Institut Laue-Langevin (ILL), under the supervision of Dr. Bratislav Lukić, as the second-generation successor to `Structural_Continuity_Analysis_Gen1_ArchimedeanSpiral.ipynb`. The code of both generations is available at https://github.com/genariturri/winding-detection-algorithms. For inquiries or further information, please reach out via email (genariturri@gmail.com) or connect on LinkedIn (https://www.linkedin.com/in/genar-iturri-bello/) or GitHub (https://github.com/genariturri).

## Phase 1: Setup and Geometry Detection

### Cell 1: Imports and Input Data

Imports every library used in this notebook and defines the input data. **Before running, update the two settings at the end of this cell:** `DATA_DIR`, the folder that contains the slices, and `SLICE_FILES`, the slices to analyse, each given as a label used in the output and the name of its file inside `DATA_DIR`. Each slice is a single 2D TIFF image of a denoised reconstruction and is read individually, unlike Generation 1, which reads one multi-page TIFF stack.

In [ ]:
import os
import numpy as np
import tifffile
from scipy.ndimage import gaussian_filter, gaussian_filter1d, map_coordinates
from skimage.transform import warp_polar
from skimage.filters import threshold_otsu
from skimage.measure import label
import matplotlib.pyplot as plt

# --- Input data: update these to match your own filesystem ---
# Folder that contains the denoised slices, each stored as a single 2D TIFF
# image ("." is the folder of this notebook).
DATA_DIR = "/path/to/your/slices"  # <-- update this to your own denoised-slices folder

# Slices analysed in Phase 5: {label used in the output: file name inside DATA_DIR}
SLICE_FILES = {
    "Slice 200": "slice_0200_n2i.tif",
    "Slice 350": "slice_0350_n2i.tif",
    "Slice 500": "slice_0500_n2i.tif",
    "Slice 650": "slice_0650_n2i.tif",
}


### Cell 2: Radial Sampling — the Basic Building Block

`radial_profile` samples image intensity along a single straight ray cast from a center point outward, at a fixed angle, using bilinear interpolation (`scipy.ndimage.map_coordinates`, order=1) since the sampled (x, y) positions essentially never fall exactly on an integer pixel. Every other geometry-detection function in this notebook is built on top of this one primitive.

In [ ]:
def radial_profile(image, cy, cx, angle_rad, r_max, r_min=0):
    """
    Samples the image intensity along a straight ray cast from the centre
    (cy, cx) outward, at a fixed angle, from r_min to r_max.

    This is the basic building block used by every other function in this
    file: any measurement that needs to "look outward from the centre" in
    a given direction goes through this function.
    """
    # Build the list of radii we want to sample along this ray.
    radii = np.arange(r_min, r_max)

    # Convert (angle, radius) into actual pixel coordinates (x, y) using
    # standard polar-to-Cartesian conversion.
    ys = cy + radii * np.sin(angle_rad)
    xs = cx + radii * np.cos(angle_rad)

    # Some of these coordinates might fall outside the image bounds
    # (e.g. if the ray points toward a corner); discard those points.
    valid = (ys >= 0) & (ys < image.shape[0]) & (xs >= 0) & (xs < image.shape[1])
    radii, ys, xs = radii[valid], ys[valid], xs[valid]

    # (x, y) coordinates are almost never exact integer pixel positions,
    # so we read the image intensity there using bilinear interpolation
    # (order=1): each sampled value is a weighted blend of its 4 nearest
    # neighbouring pixels.
    profile = map_coordinates(image, [ys, xs], order=1)
    return radii, profile


### Cell 3: Center and Outer-Radius Detection

`detect_max_radius` finds the battery's true center and outer boundary radius by iteratively fitting a circle to the strongest intensity edge around the full circumference, rather than relying on a single fixed intensity threshold (which is less robust when contrast varies from slice to slice):

1. A coarse initial center is estimated from a simple threshold mask.
2. At each of several iterations: a ray is cast at many angles (`radial_profile`), the radius of the steepest intensity gradient along each ray is taken as that angle's boundary estimate, angles whose estimate is far from the median are discarded as outliers, and a first-order harmonic `r(θ) ≈ R0 + dx·cos(θ) + dy·sin(θ)` is fit by least squares to the remaining points — the same linear centering-correction model Generation 1 uses in `refine_center`, but here it drives a full iterative convergence loop rather than a single one-shot correction.
3. The final outer radius is the median of the boundary measurements at the converged center.

In [ ]:
def detect_max_radius(image, n_iterations=4, angle_step_deg=2, inner_margin=0.3,
                       outlier_tolerance=60):
    """
    Detects the true centre of the spiral and its outer radius (the
    physical boundary of the battery winding), by locating the strongest
    intensity edge around the full circumference -- rather than relying on
    a fixed intensity threshold, which is less reliable when contrast
    varies from image to image.
    """
    H, W = image.shape

    # --- Step 1: rough initial centre guess ---
    # Blur the image to remove fine texture noise, then threshold at 40%
    # of the image's own dynamic range (relative, not an absolute value,
    # since brightness normalisation can vary between slices).
    smoothed = gaussian_filter(image, sigma=5)
    thresh = image.min() + 0.4 * (image.max() - image.min())
    yy, xx = np.where(smoothed > thresh)

    # The centroid of the thresholded region is our first, coarse
    # estimate of the centre.
    cy, cx = float(np.mean(yy)), float(np.mean(xx))

    r_max_search = int(min(H, W) / 2 * 0.98)
    angles = np.arange(0, 360, angle_step_deg)

    # --- Step 2: iteratively refine the centre and measure the boundary ---
    # Each iteration: (a) measure where the boundary is at many angles,
    # using the current centre estimate, then (b) fit a correction to the
    # centre so that the boundary looks as circular as possible, and
    # repeat. After a few iterations this converges to the true centre.
    for _ in range(n_iterations):
        boundary_radii = []
        for a_deg in angles:
            # Sample a ray outward and smooth it slightly, then look at
            # its gradient (rate of change). The boundary of the battery
            # produces the single strongest brightness jump in the image,
            # so the radius with the largest |gradient| is our boundary
            # estimate at this angle.
            radii, profile = radial_profile(image, cy, cx, np.deg2rad(a_deg), r_max_search)
            profile_smooth = gaussian_filter1d(profile, sigma=4)
            gradient = np.abs(np.gradient(profile_smooth))

            # Zero out the innermost part of the search range so the
            # bright central blob near the centre can never be mistaken
            # for the outer boundary.
            gradient[:int(r_max_search * inner_margin)] = 0
            boundary_radii.append(radii[np.argmax(gradient)])

        boundary_radii = np.array(boundary_radii, dtype=float)

        # Discard angles whose measured boundary radius is far from the
        # median -- these are likely spurious detections (e.g. a mounting
        # artefact), and would otherwise bias the centre correction below.
        median_r = np.median(boundary_radii)
        keep = np.abs(boundary_radii - median_r) < outlier_tolerance

        # If the assumed centre is off by a small offset (dx, dy), the
        # radius of a genuinely circular boundary measured from that
        # off-centre point follows, to first order,
        #   r(theta) ~= R0 + dx*cos(theta) + dy*sin(theta)
        # We fit this linear model by ordinary least squares to find the
        # centre correction (dx, dy) that makes the boundary most circular.
        theta = np.deg2rad(angles[keep])
        design_matrix = np.stack([np.ones_like(theta), np.cos(theta), np.sin(theta)], axis=1)
        coef, *_ = np.linalg.lstsq(design_matrix, boundary_radii[keep], rcond=None)
        R0, dx, dy = coef

        # Apply the correction and loop again with the improved centre.
        cy, cx = cy + dy, cx + dx

    # Final radius: the median of the (outlier-filtered) boundary
    # measurements taken at the last, converged centre.
    max_radius = float(np.median(boundary_radii[keep]))
    return (cy, cx), max_radius


### Cell 4: Automatic Inner-Radius (Hollow-Core) Detection

`detect_spiral_inner_radius` finds where the winding actually begins — i.e. where the hollow, structureless core ends — automatically, rather than assuming a fixed value as Generation 1 does (`min_material_radius=190` there). It measures, at many angles, how much a background-subtracted radial profile varies *across angles* at each radius: inside the hollow core this is just flat sensor noise, while once real winding structure begins, different angles cross different points of the alternating layer pattern and the variation rises sharply to a plateau. The inner radius is taken as the first point where this variation crosses halfway between its noise floor and its plateau, plus a small safety margin.

In [ ]:
def detect_spiral_inner_radius(image, center_yx, max_radius, n_test_angles=180,
                                baseline_sigma=15, threshold_fraction=0.5,
                                safety_margin=15):
    """
    Automatically finds the radius at which the spiral winding actually
    begins -- i.e. where the hollow, structureless core ends -- instead of
    assuming a fixed value. This matters because the hollow core has no
    winding to trace, and treating it as if it did produces meaningless
    results.
    """
    cy, cx = center_yx
    angles = np.linspace(0, 2 * np.pi, n_test_angles, endpoint=False)
    r_max_int = int(max_radius)

    # For many angles, compute a "residual" profile: the raw radial
    # profile minus a heavily smoothed version of itself. This keeps the
    # fine, winding-induced oscillation while removing the slow overall
    # brightness trend.
    residual_stack = []
    for a in angles:
        radii, profile = radial_profile(image, cy, cx, a, r_max_int)
        profile_smooth = gaussian_filter1d(profile, sigma=1.5)
        baseline = gaussian_filter1d(profile, sigma=baseline_sigma)
        residual_stack.append(profile_smooth - baseline)
    residual_stack = np.array(residual_stack)

    # At each radius, measure how much the residual varies ACROSS angles.
    # Inside the hollow core, this is just flat sensor noise (low and
    # constant); once we reach the real winding, different angles cross
    # different points of the alternating layer pattern, so the variation
    # rises sharply and settles into a plateau.
    amplitude = gaussian_filter1d(np.std(residual_stack, axis=0), sigma=5)

    # Estimate the "noise floor" from deep inside the core (r < 40px,
    # certainly still hollow), and the "plateau" level from a range that
    # is certainly already inside real winding structure (150-400px).
    noise_floor = np.median(amplitude[:40])
    plateau = np.median(amplitude[150:min(400, len(amplitude))])

    # The inner radius is the first point where the amplitude crosses
    # half-way between the noise floor and the plateau -- i.e. where the
    # signal has clearly transitioned from "hollow" to "winding".
    threshold = noise_floor + threshold_fraction * (plateau - noise_floor)
    above = np.where(amplitude > threshold)[0]
    inner_radius = float(above[0]) if len(above) else 0.0

    # A small safety margin pushes the boundary slightly further out, to
    # avoid including the transition zone itself (which is not yet clean
    # winding signal).
    return inner_radius + safety_margin


## Phase 2: Geometry-Independent Polar Unwrapping

### Cell 5: Polar Binarization Without a Pitch Model

`build_polar_binary` is the key departure from Generation 1. It resamples the image onto a plain (angle, radius) grid (`skimage.transform.warp_polar`) — exactly as Generation 1's `spiral_layer_continuity` does — but **skips the spiral-pitch correction step entirely**. No pitch is measured, and no assumption is made about how fast the radius should grow with angle. As a direct consequence, each winding appears in this unwrapped image as a continuous **diagonal** band rather than a horizontal one; the diagonal is simply whatever shape the real winding actually has, uncorrupted by a constant-pitch assumption that might not fit it.

Background subtraction and adaptive (radius-dependent) Otsu thresholding are then applied exactly as in Generation 1 — a wide Gaussian blur along the radius axis estimates the local brightness trend, and the threshold is computed independently in overlapping windows along the radius and interpolated between them, because contrast is not uniform with radius in this data.

In [ ]:
def build_polar_binary(image, center_yx, max_radius, min_radius, n_angles=1440,
                        background_sigma=15, threshold_window=40, threshold_step=10):
    """
    Converts the image to polar coordinates WITHOUT assuming any spiral
    pitch, and binarises it using a local (radius-adaptive) Otsu
    threshold. Each real winding appears as a continuous DIAGONAL band
    (not horizontal), because no pitch correction is applied here -- this
    is intentional: we do not want the algorithm to depend on any
    constant-pitch model of the spiral, since that assumption is what
    made earlier versions of this pipeline fragile.
    """
    cy, cx = center_yx
    r_max_int = int(max_radius)

    # Resample the image onto a (angle, radius) grid. Output row i,
    # column j corresponds to angle i*(360/n_angles) degrees, radius j.
    polar = warp_polar(image, center=center_yx, radius=r_max_int, output_shape=(n_angles, r_max_int))

    # Remove the slow background brightness trend (e.g. the gradual
    # falloff near the bright outer casing) by subtracting a version of
    # the image smoothed heavily along the radial axis only. What remains
    # is the fine, winding-to-winding contrast we actually care about.
    background = gaussian_filter1d(polar, sigma=background_sigma, axis=1)
    residual = polar - background

    # Compute an Otsu threshold independently within many overlapping
    # windows along the radius (each threshold_window px wide, sampled
    # every threshold_step px), then interpolate between them to get a
    # smooth threshold value at every radius. This is necessary because
    # contrast is not uniform with radius in this data: a single global
    # threshold would be dominated by whichever radial region has the
    # widest intensity spread.
    sampled_radii = list(range(0, r_max_int, threshold_step))
    sampled_thresholds = []
    for c in sampled_radii:
        c0, c1 = max(0, c - threshold_window // 2), min(r_max_int, c + threshold_window // 2)
        window = residual[:, c0:c1]
        # threshold_otsu requires some contrast in the window; if the
        # window is completely flat (e.g. far outside the sample), fall
        # back to its mean as a harmless placeholder threshold.
        sampled_thresholds.append(threshold_otsu(window) if window.max() > window.min() else window.mean())
    threshold_profile = np.interp(np.arange(r_max_int), sampled_radii, sampled_thresholds)

    # Binarise: True where the residual exceeds the LOCAL threshold for
    # that specific radius.
    binary = residual > threshold_profile[np.newaxis, :]

    # The hollow central core has no winding structure at all; force it
    # to False so it can never be mistaken for a winding later on.
    binary[:, :int(min_radius)] = False
    return polar, residual, binary


## Phase 3: Winding Extraction via Connected Components

### Cell 6: Tracing Each Winding as a Connected Region

`trace_windings_via_components` is the second key departure from Generation 1. Rather than detecting intensity peaks angle-by-angle and linking each one to the next — which is fragile, since a single angle with one extra or one missing peak makes the link jump onto the wrong, neighboring winding — this function labels the **whole 2D shape** of each winding at once (`skimage.measure.label`, 8-connectivity, since the winding bands are diagonal rather than axis-aligned).

As in Generation 1's `detect_discontinuities`, the angular axis is cyclically padded before labeling so a winding crossing the 0°/360° seam is not artificially split in two, and the padded copy is used only to resolve the seam — the genuine, unpadded measurement is preferred wherever both exist. A labeled region is only accepted as a real winding if:

- its area is at least `min_area` (discards small noise regions), and
- it spans at least `min_angular_coverage_fraction` of the full circle (discards fragments that are not a complete turn).

For every angle where an accepted region is present, its column centroid gives that winding's radial position at that angle — this sparse `{angle: radius}` mapping is the function's output for each winding.

In [ ]:
def trace_windings_via_components(binary, n_angles, min_angular_coverage_fraction=0.9,
                                   min_area=200):
    """
    Finds each winding as a SINGLE connected 2D region (a diagonal band in
    the binarised angle-vs-radius image), instead of detecting intensity
    peaks angle-by-angle and linking them one to the next. Peak-linking
    turned out to be fragile: a single angle with one extra or one
    missing peak makes the link jump onto the wrong, neighbouring
    winding. Connected-component labelling avoids this failure mode
    entirely, because it considers each winding's whole 2D shape at once
    rather than a sequence of independent 1D measurements.
    """
    # A winding may cross the point where the angle wraps from 360 back
    # to 0. To let connected-component labelling handle this correctly
    # without special-case logic, we pad a quarter of a full turn onto
    # both ends of the angular axis -- this moves the true seam safely
    # into the interior of the extended image, away from any edge.
    pad = n_angles // 4
    cyclic = np.concatenate([binary[-pad:, :], binary, binary[:pad, :]], axis=0)

    # Label every connected group of True pixels as one region.
    # connectivity=2 means 8-connectivity (each pixel is adjacent to all
    # 8 neighbours, including diagonals) -- necessary because the winding
    # bands are diagonal, not axis-aligned.
    labels = label(cyclic, connectivity=2)

    tracks = []
    for lbl in range(1, labels.max() + 1):
        mask = labels == lbl

        # Discard small regions: these are almost certainly noise, not a
        # genuine winding.
        if mask.sum() < min_area:
            continue

        # A genuine, unbroken winding must span (almost) the full circle.
        # If this region only covers a small angular range, it is a
        # fragment, not a complete turn -- discard it.
        rows_present = np.where(mask.any(axis=1))[0]
        angular_extent = rows_present.max() - rows_present.min()
        if angular_extent < min_angular_coverage_fraction * n_angles:
            continue

        # For every angular row, take the centroid (mean column index) of
        # the pixels belonging to this region -- that is the winding's
        # radial position at that angle.
        #
        # Because of the padding, a given real angle can appear twice
        # (once in the padded copy, once in the main image). We record
        # both ("by_angle_any", first-seen value) but prefer whichever
        # value came from the main, unpadded region ("by_angle_central"),
        # since that one is not affected by the artificial seam -- this
        # avoids a small visible glitch exactly at the 0/360 boundary.
        by_angle_any = {}
        by_angle_central = {}
        for row in range(mask.shape[0]):
            real_row = (row - pad) % n_angles
            cols = np.where(mask[row, :])[0]
            if len(cols) == 0:
                continue
            r_center = cols.mean()
            if real_row not in by_angle_any:
                by_angle_any[real_row] = r_center
            if pad <= row < pad + n_angles:
                by_angle_central[real_row] = r_center

        # Central-region values (if present) overwrite the fallback ones.
        by_angle = {**by_angle_any, **by_angle_central}

        # After deduplication, re-check that we still have (near-)full
        # angular coverage before accepting this as a real winding.
        if len(by_angle) < min_angular_coverage_fraction * n_angles:
            continue
        tracks.append(by_angle)
    return tracks


### Cell 7: Filling Small Gaps

`densify_track` converts one winding's sparse `{angle: radius}` mapping into a dense array covering every angle, filling small gaps by **circular** interpolation (`period=n_angles`) so the fill wraps correctly across the 0°/360° boundary instead of treating it as a discontinuity.

In [ ]:
def densify_track(by_angle, n_angles):
    """
    Converts a sparse {angle_index: radius} mapping for one winding into a
    dense array covering every angle from 0 to n_angles-1, filling any
    small gaps by interpolation.

    Uses CIRCULAR interpolation (period=n_angles), so that the
    interpolation wraps correctly across the 0/360 degree boundary
    instead of treating it as a discontinuity.
    """
    idxs = np.array(sorted(by_angle.keys()))
    radii = np.array([by_angle[i] for i in idxs])
    full_idx = np.arange(n_angles)
    full_r = np.interp(full_idx, idxs, radii, period=n_angles)
    return full_idx, full_r


## Phase 4: Visualization

### Cell 8: Visual Verification

`plot_sanity_check` draws the original tomography image with the detected outer boundary (blue circle), detected inner radius (dashed cyan circle), and every accepted winding as a continuous colored curve, converted back from (angle, radius) into real (x, y) image coordinates. This is the notebook's main correctness check: each curve should sit exactly on top of one real physical winding, continuously, all the way around.

In [ ]:
def plot_sanity_check(image, center_yx, max_radius, min_radius, tracks, n_angles, title=""):
    """
    Visual verification tool: draws the original tomography image with
    the detected outer boundary (blue circle), the detected inner radius
    (dashed cyan circle), and every detected winding as a continuous
    coloured curve, converted back from (angle, radius) into real (x, y)
    image coordinates.
    """
    cy, cx = center_yx
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(image, cmap='gray')

    # Outer and inner radius, drawn as reference circles.
    ax.add_patch(plt.Circle((cx, cy), max_radius, color='blue', fill=False, linewidth=2))
    ax.add_patch(plt.Circle((cx, cy), min_radius, color='cyan', fill=False, linewidth=1, linestyle='--'))

    cmap = plt.get_cmap('tab20')
    angles = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)

    # Draw each detected winding, converting its (angle, radius) samples
    # back into Cartesian (x, y) coordinates so it overlays correctly on
    # top of the original circular image.
    for i, t in enumerate(tracks):
        full_idx, full_r = densify_track(t, n_angles)
        a = angles[full_idx]
        x = cx + full_r * np.cos(a)
        y = cy + full_r * np.sin(a)
        ax.plot(x, y, '-', color=cmap(i % 20), linewidth=1.3)

    ax.set_title(f"{title}  ({len(tracks)} windings)")
    ax.axis('off')
    plt.show()


## Phase 5: Demonstration Across Multiple Slices

### Cell 9: Running the Full Pipeline on Sample Slices

Runs the complete pipeline — geometry detection, polar binarization, winding extraction, and visual verification — independently on every slice listed in `SLICE_FILES` (four slices spanning the volume, in the report), **without changing any parameter between them**. This is the test used to confirm the algorithm generalizes across slices rather than being tuned to just one. The slices are read from `DATA_DIR`; both settings are in Cell 1.

As noted in the notebook's introduction (Limitations), this cell reports how many windings were traced per slice but does not yet classify or locate defects the way Generation 1's batch loop does — extending it to do so (and to loop over a full stack rather than a list of named files) is the natural next step for whoever picks this notebook up next.

In [ ]:
# ---- EXECUTION ----
# Runs the full pipeline (geometry detection -> polar binarisation ->
# winding extraction -> visual sanity check) independently on every slice
# listed in SLICE_FILES (Cell 1), WITHOUT changing any parameter between
# them -- this is the test used to confirm the algorithm generalises across
# slices instead of being tuned to just one.
n_angles = 1440

# Stop early, with a clear message, if a slice is not where Cell 1 says it is.
missing = [f for f in SLICE_FILES.values() if not os.path.isfile(os.path.join(DATA_DIR, f))]
if missing:
    raise FileNotFoundError(
        f"Not found in DATA_DIR ({DATA_DIR}): {', '.join(missing)}. "
        "Update DATA_DIR and SLICE_FILES in Cell 1."
    )

for title, filename in SLICE_FILES.items():
    image = tifffile.imread(os.path.join(DATA_DIR, filename)).astype(np.float32)

    # Stage 1: geometry (centre, outer radius, inner radius).
    center, max_radius = detect_max_radius(image)
    min_radius = detect_spiral_inner_radius(image, center, max_radius)
    print(f"{title}: center={center}, inner radius={min_radius:.1f}px, outer radius={max_radius:.1f}px")

    # Stage 2 + 3: polar binarisation and winding extraction.
    _, _, binary = build_polar_binary(image, center, max_radius, min_radius, n_angles=n_angles)
    tracks = trace_windings_via_components(binary, n_angles)
    print(f"  Windings traced: {len(tracks)}")

    # Stage 4: visual verification.
    plot_sanity_check(image, center, max_radius, min_radius, tracks, n_angles, title=title)
